# What this notebook is doing (HRRR precip GIF)

This notebook makes a precipitation GIF from HRRR for a selected time window over the Skagit basin.
The main point is to get a quick visual of how the precip evolves through an event instead of just looking at summary numbers.

It opens the HRRR data, builds a mask for the Skagit boundary, pulls the precip field, and then saves one frame at each selected timestep.
Right now it is set up to use a 24-hour rolling accumulation and 6-hour frame spacing, so the GIF shows a smoother event-scale view rather than noisy hourly snapshots.

At the end it writes both the individual PNG frames and the final GIF.


In [ ]:
# ============================================================
# HRRR precip GIF (Skagit boundary) for 2021-11-10 → 2021-11-17
# Mask: geopandas + regionmask 
# Data: tp (kg/m^2 == mm)
#
# Fixes:
# - De-duplicate time (HRRR zarr can contain duplicate timestamps)
# - Optional rolling accumulation (default 24h) to visualize "total precip"
# - Fixed color scale (constant across frames)
# - Reliable GIF playback speed using PIL writer (duration in milliseconds)
# ============================================================

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import imageio.v2 as imageio
from pathlib import Path

# ---------------- Settings ----------------
BASE = Path("/data0/balaji24/data/weather_data")
BOUNDARY_GEO = Path("../data/GIS/SkagitBoundary.json")

START = "2021-11-10 00:00:00"
END   = "2021-11-17 23:00:00"

FRAME_FREQ = "6h"          # "1h" for hourly frames
ROLLING_HOURS = 24         # 0 for instantaneous; 24 for 24h rolling total

USE_AUTO_SCALE = False     # True to auto-scale vmax from data quantile
AUTO_Q = 0.995

FORCE_VMIN = 0
FORCE_VMAX = 10            # try 10, 25, 50, 80, 250

FRAME_DURATION_SEC = 1   # seconds per frame (reliably written via PIL below)

OUTDIR = Path("./_hrrr_gif_frames_20211110_20211117")
GIF_PATH = Path("./hrrr_20211110_20211117.gif")

# ---------------- Helpers ----------------
def safe_open_zarr(p: Path):
    try:
        return xr.open_zarr(p, consolidated=True)
    except Exception:
        return xr.open_zarr(p, consolidated=False)

def find_lat_lon_names(ds):
    lat_name = next((n for n in ["lat", "latitude", "XLAT", "XLAT_M", "lat2d"] if n in ds), None)
    lon_name = next((n for n in ["lon", "longitude", "XLONG", "XLONG_M", "lon2d"] if n in ds), None)
    return lat_name, lon_name

def _to_2d(arr):
    if "time" in arr.dims:
        arr = arr.isel(time=0)
    for d in list(arr.dims):
        if arr.sizes[d] == 1:
            arr = arr.isel({d: 0})
    while arr.ndim > 2:
        extra = [d for d in arr.dims if d not in ("x", "y", "lon", "lat")]
        if not extra:
            break
        arr = arr.isel({extra[0]: 0})
    return arr

def ensure_time_sorted_unique_da(da: xr.DataArray) -> xr.DataArray:
    if "time" not in da.dims:
        return da
    da = da.sortby("time")
    t = pd.to_datetime(da["time"].values)
    _, idx = np.unique(t, return_index=True)
    return da.isel(time=np.sort(idx))

def ensure_time_sorted_unique_ds(ds: xr.Dataset) -> xr.Dataset:
    if "time" not in ds.dims:
        return ds
    ds = ds.sortby("time")
    t = pd.to_datetime(ds["time"].values)
    _, idx = np.unique(t, return_index=True)
    return ds.isel(time=np.sort(idx))

def open_hrrr_window(t0: str, t1: str) -> xr.Dataset:
    t0 = pd.Timestamp(t0)
    t1 = pd.Timestamp(t1)
    months = pd.period_range(t0.to_period("M"), t1.to_period("M"), freq="M")

    paths = []
    for per in months:
        z = BASE / f"{per.year:04d}-{per.month:02d}_HRRR_data.zarr"
        if z.exists():
            paths.append(z)
    if not paths:
        raise FileNotFoundError("No HRRR zarr months found for that window.")

    dsets = [safe_open_zarr(p) for p in paths]
    ds = xr.concat(dsets, dim="time", join="outer", coords="minimal", compat="override").sortby("time")
    ds = ds.sel(time=slice(str(t0), str(t1)))
    ds = ensure_time_sorted_unique_ds(ds)
    return ds

def pick_precip_var(ds: xr.Dataset) -> str:
    if "tp" in ds.data_vars:
        return "tp"
    if "apcp" in ds.data_vars:
        return "apcp"
    raise KeyError(f"No precip var found. data_vars={list(ds.data_vars)}")

def build_skagit_boundary_mask(ds: xr.Dataset) -> xr.DataArray:
    import geopandas as gpd
    import regionmask

    if not BOUNDARY_GEO.exists():
        raise FileNotFoundError(f"Boundary geojson not found: {BOUNDARY_GEO}")

    skagit_gdf = gpd.read_file(BOUNDARY_GEO).to_crs("EPSG:4326")
    geom = skagit_gdf.geometry.union_all()

    lat_name, lon_name = find_lat_lon_names(ds)
    if not (lat_name and lon_name):
        raise KeyError("Could not find latitude/longitude in HRRR dataset.")

    lon = _to_2d(ds[lon_name])
    lat = _to_2d(ds[lat_name])

    regs = regionmask.Regions(
        outlines=[geom],
        names=["Skagit"],
        numbers=[0],
        name="SkagitBoundary",
    )
    rid = regs.mask(lon, lat)  # 0 inside, NaN outside
    return xr.where(rid == 0, True, False).astype(bool)

# ---------------- Main ----------------
OUTDIR.mkdir(parents=True, exist_ok=True)

ds = open_hrrr_window(START, END)
pvar = pick_precip_var(ds)

mask = build_skagit_boundary_mask(ds)
da = ds[pvar].astype("float32")

if ROLLING_HOURS and ROLLING_HOURS > 0:
    da = da.rolling(time=ROLLING_HOURS, min_periods=1).sum()

da = ensure_time_sorted_unique_da(da)

if USE_AUTO_SCALE:
    vmin = 0.0
    vmax = float(da.where(mask).quantile(AUTO_Q).compute().values)
else:
    vmin, vmax = float(FORCE_VMIN), float(FORCE_VMAX)

print(f"[scale] vmin={vmin:.3f}, vmax={vmax:.3f} (ROLLING_HOURS={ROLLING_HOURS})")

frame_times = pd.date_range(pd.Timestamp(START), pd.Timestamp(END), freq=FRAME_FREQ)
available = pd.to_datetime(da["time"].values)
available_set = set(available.values)
frame_times = [t for t in frame_times if t.to_datetime64() in available_set]
if not frame_times:
    raise ValueError("No frame times matched exactly. Try FRAME_FREQ='1h'.")

lat_name, lon_name = find_lat_lon_names(ds)
LAT = _to_2d(ds[lat_name]).values
LON = _to_2d(ds[lon_name]).values

frame_paths = []

for i, t in enumerate(frame_times, start=1):
    snap = da.sel(time=t)
    if "time" in snap.dims and snap.sizes["time"] > 1:
        snap = snap.isel(time=0)

    snap_plot = snap.where(mask)

    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.pcolormesh(LON, LAT, snap_plot.values, shading="auto", vmin=vmin, vmax=vmax, cmap="plasma")
    ax.set_title(f"initial time of forecast: {t:%Y-%m-%d %H:%M:%S}")
    ax.set_xlabel("longitude")
    ax.set_ylabel("latitude (degrees_north)")
    cb = fig.colorbar(im, ax=ax)
    cb.set_label("total precip (mm)")

    fig.tight_layout()
    fp = OUTDIR / f"frame_{i:04d}.png"
    fig.savefig(fp, dpi=150)
    plt.close(fig)
    frame_paths.append(fp)

# Reliable GIF writer with per-frame duration (ms)
from PIL import Image

duration_ms = int(FRAME_DURATION_SEC * 1000)
pil_frames = [Image.open(p).convert("P", palette=Image.Palette.ADAPTIVE) for p in frame_paths]
pil_frames[0].save(
    GIF_PATH,
    save_all=True,
    append_images=pil_frames[1:],
    duration=duration_ms,
    loop=0,
    optimize=False,
)

print("Saved GIF ->", GIF_PATH.resolve())
print("Saved frames ->", OUTDIR.resolve())
print("Frames:", len(frame_paths), "Var:", pvar, "Freq:", FRAME_FREQ, "SecPerFrame:", FRAME_DURATION_SEC)

[scale] vmin=0.000, vmax=10.000 (ROLLING_HOURS=24)
Saved GIF -> /home/balaji24/skagit-met/analysis/hrrr_20211110_20211117.gif
Saved frames -> /home/balaji24/skagit-met/analysis/_hrrr_gif_frames_20211110_20211117
Frames: 32 Var: tp Freq: 6h SecPerFrame: 1
